### Sentences

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

model_id = "embed_model"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).eval().cuda()


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def encode(texts, batch_size=32):
    all_embs = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        batch = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            out = model(**batch)
            emb = mean_pooling(out.last_hidden_state, batch["attention_mask"])
            emb = F.normalize(emb, p=2, dim=1)

        all_embs.append(emb.cpu())

    return torch.cat(all_embs, dim=0)

texts = [

# countries / geography
"Paris is the capital of France.",
"France has Paris as its capital.",
"The capital city of France is Paris.",
"Berlin is the capital of Germany.",
"Germany is a country in Europe.",
"Berlin is the largest city in Germany.",
"Cairo is the capital of Egypt.",
"Egypt is located in North Africa.",
"The Nile river flows through Egypt.",
"Tokyo is the capital of Japan.",
"Japan is an island country.",
"Mount Fuji is in Japan.",
"London is the capital of the United Kingdom.",
"The UK is a country in Europe.",
"Big Ben is a landmark in London.",
"Madrid is the capital of Spain.",
"Rome is the capital of Italy.",
"Canada is a large country in North America.",
"Beijing is the capital of China.",
"India is the most populous democracy.",

# animals
"A dog is an animal.",
"Dogs are loyal animals.",
"A cat is an animal.",
"Cats like to sleep a lot.",
"A lion is a wild animal.",
"Lions live in the savannah.",
"A tiger is a large predator.",
"Tigers have orange and black stripes.",
"Wolves live in packs.",
"Wolves hunt together.",
"Elephants are large mammals.",
"Elephants have long trunks.",
"Horses are strong animals.",
"Horses can run very fast.",
"Cows are farm animals.",
"Birds can fly in the sky.",
"Eagles fly very high.",
"Fish live in water.",
"Sharks are dangerous fish.",
"Dolphins are intelligent animals.",

# tech companies
"Microsoft is a technology company.",
"Google is a technology company.",
"Apple is a technology company.",
"Amazon is a technology company.",
"Meta builds social media platforms.",
"Nvidia makes graphics processors.",
"OpenAI develops artificial intelligence.",
"Tesla builds electric cars.",
"Google develops search engines.",
"Apple designs smartphones.",
"Microsoft develops Windows.",
"Amazon runs cloud services.",
"Nvidia builds GPUs.",
"OpenAI trains large language models.",

# programming / AI
"I love machine learning.",
"Machine learning finds patterns in data.",
"Deep learning uses neural networks.",
"Neural networks learn from data.",
"Transformers use self-attention.",
"Attention mechanisms help models focus.",
"Python is a programming language.",
"Python is widely used for AI.",
"JavaScript runs in web browsers.",
"Rust is a systems programming language.",
"Large language models generate text.",
"AI models can summarize documents.",
"Neural networks contain many parameters.",
"Transformers are used in modern NLP.",

# food
"I am eating an apple.",
"She is eating a banana.",
"They are cooking pasta.",
"He is baking bread.",
"We are making pizza.",
"People drink coffee in the morning.",
"Tea is a popular drink.",
"Chocolate tastes sweet.",
"Strawberries are sweet fruits.",
"Oranges contain vitamin C.",
"Many people eat rice every day.",
"Soup is warm and comforting.",
"Cheese is made from milk.",
"Salad contains fresh vegetables.",

# sports
"Lionel Messi plays football.",
"Cristiano Ronaldo plays football.",
"Football is the most popular sport.",
"Basketball is played with a ball.",
"NBA players play basketball.",
"Tennis is played with a racket.",
"Running is a popular sport.",
"Marathons require endurance.",
"Swimming is a water sport.",
"Football teams compete in leagues.",
"Players train every day.",
"Sports improve physical fitness.",

# transportation
"He is driving a car.",
"She is riding a bicycle.",
"They are traveling by train.",
"Airplanes fly in the sky.",
"Ships travel across the ocean.",
"Cars move on roads.",
"Buses transport passengers.",
"Trains run on railways.",
"Airplanes carry passengers between countries.",
"Subways run underground in cities.",
"Motorcycles are fast vehicles.",

# weather / nature
"The sky is blue.",
"The sun is very bright.",
"Rain falls from clouds.",
"Snow falls in winter.",
"The wind is blowing.",
"Storms bring heavy rain.",
"Clouds move across the sky.",
"Thunder follows lightning.",
"The weather is cold today.",
"The weather is warm and sunny.",

# space
"The Earth orbits the Sun.",
"The Moon orbits the Earth.",
"Mars is a planet.",
"Jupiter is the largest planet.",
"Satellites orbit the Earth.",
"Astronauts travel to space.",
"The solar system contains many planets.",
"Space is vast and mysterious.",
"Rockets launch into space.",
"Telescopes observe distant galaxies.",

# emotions / human activity
"I feel very happy today.",
"She feels sad.",
"He is excited about the trip.",
"They are celebrating a victory.",
"People enjoy listening to music.",
"Children love playing games.",
"Friends enjoy spending time together.",
"Music makes people happy.",
"Reading books is relaxing.",
"Traveling helps people learn about cultures.",

# science
"Water boils at one hundred degrees Celsius.",
"Gravity pulls objects toward Earth.",
"Light travels very fast.",
"Electricity powers many devices.",
"Atoms are the building blocks of matter.",
"Energy can change forms.",
"Physics studies matter and energy.",
"Biology studies living organisms.",
"Chemistry studies reactions between substances.",
]

embeddings = encode(texts)

with open("../../embedding-projector-standalone/oss_data/embeddings.tsv", "w", encoding="utf-8") as f_emb, \
        open("../../embedding-projector-standalone/oss_data/metadata.tsv", "w", encoding="utf-8") as f_meta:

    for text, emb in zip(texts, embeddings):
        f_emb.write("\t".join(str(x.item()) for x in emb) + "\n")
        f_meta.write(text.replace("\n", " ") + "\n")

print("saved: embeddings.tsv, metadata.tsv")

### words

In [2]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

model_id = "embed_model"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).eval().cuda()


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def encode(texts, batch_size=32):
    all_embs = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        batch = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=8,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            out = model(**batch)
            emb = mean_pooling(out.last_hidden_state, batch["attention_mask"])
            emb = F.normalize(emb, p=2, dim=1)

        all_embs.append(emb.cpu())

    return torch.cat(all_embs, dim=0)

items = [

# animals_big
("lion","animals_big"),
("tiger","animals_big"),
("elephant","animals_big"),
("giraffe","animals_big"),
("bear","animals_big"),
("horse","animals_big"),
("cow","animals_big"),
("zebra","animals_big"),
("kangaroo","animals_big"),
("leopard","animals_big"),

# animals_small
("cat","animals_small"),
("rabbit","animals_small"),
("fox","animals_small"),
("goat","animals_small"),
("sheep","animals_small"),
("deer","animals_small"),
("monkey","animals_small"),
("panda","animals_small"),
("dog","animals_small"),
("wolf","animals_small"),

# birds
("bird","birds"),
("eagle","birds"),
("hawk","birds"),
("sparrow","birds"),
("pigeon","birds"),
("parrot","birds"),
("owl","birds"),
("falcon","birds"),
("penguin","birds"),
("flamingo","birds"),

# sea_animals
("fish","sea_animals"),
("shark","sea_animals"),
("whale","sea_animals"),
("dolphin","sea_animals"),
("octopus","sea_animals"),
("crab","sea_animals"),
("lobster","sea_animals"),
("seal","sea_animals"),
("jellyfish","sea_animals"),
("shrimp","sea_animals"),

# fruits
("apple","fruits"),
("banana","fruits"),
("orange","fruits"),
("grape","fruits"),
("mango","fruits"),
("pineapple","fruits"),
("strawberry","fruits"),
("peach","fruits"),
("pear","fruits"),
("watermelon","fruits"),

# vegetables
("carrot","vegetables"),
("potato","vegetables"),
("tomato","vegetables"),
("onion","vegetables"),
("garlic","vegetables"),
("pepper","vegetables"),
("cucumber","vegetables"),
("broccoli","vegetables"),
("spinach","vegetables"),
("lettuce","vegetables"),

# countries
("france","countries"),
("germany","countries"),
("egypt","countries"),
("japan","countries"),
("china","countries"),
("india","countries"),
("italy","countries"),
("spain","countries"),
("canada","countries"),
("brazil","countries"),
("mexico","countries"),
("argentina","countries"),
("turkey","countries"),
("russia","countries"),
("australia","countries"),
("korea","countries"),
("thailand","countries"),
("vietnam","countries"),
("sweden","countries"),
("norway","countries"),

# cities
("paris","cities"),
("berlin","cities"),
("cairo","cities"),
("tokyo","cities"),
("london","cities"),
("rome","cities"),
("madrid","cities"),
("beijing","cities"),
("moscow","cities"),
("toronto","cities"),

# tech_companies
("google","tech_companies"),
("microsoft","tech_companies"),
("apple","tech_companies"),
("amazon","tech_companies"),
("meta","tech_companies"),
("nvidia","tech_companies"),
("openai","tech_companies"),
("tesla","tech_companies"),
("intel","tech_companies"),
("ibm","tech_companies"),

# programming_languages
("python","programming_languages"),
("javascript","programming_languages"),
("java","programming_languages"),
("rust","programming_languages"),
("go","programming_languages"),
("c","programming_languages"),
("cpp","programming_languages"),
("typescript","programming_languages"),
("kotlin","programming_languages"),
("swift","programming_languages"),

# ai_terms
("ai","ai_terms"),
("machine_learning","ai_terms"),
("deep_learning","ai_terms"),
("neural_network","ai_terms"),
("transformer","ai_terms"),
("attention","ai_terms"),
("embedding","ai_terms"),
("tokenizer","ai_terms"),
("dataset","ai_terms"),
("model","ai_terms"),

# emotions
("happy","emotions"),
("sad","emotions"),
("angry","emotions"),
("excited","emotions"),
("afraid","emotions"),
("surprised","emotions"),
("calm","emotions"),
("joy","emotions"),
("love","emotions"),
("hate","emotions"),

# weather
("sun","weather"),
("rain","weather"),
("snow","weather"),
("wind","weather"),
("storm","weather"),
("cloud","weather"),
("thunder","weather"),
("lightning","weather"),
("fog","weather"),
("hurricane","weather"),

# space
("earth","space"),
("moon","space"),
("mars","space"),
("jupiter","space"),
("saturn","space"),
("venus","space"),
("mercury","space"),
("neptune","space"),
("planet","space"),
("galaxy","space"),

# transportation
("car","transportation"),
("bus","transportation"),
("train","transportation"),
("airplane","transportation"),
("bicycle","transportation"),
("motorcycle","transportation"),
("ship","transportation"),
("subway","transportation"),
("truck","transportation"),
("tram","transportation"),

# sports
("football","sports"),
("basketball","sports"),
("tennis","sports"),
("cricket","sports"),
("baseball","sports"),
("golf","sports"),
("swimming","sports"),
("running","sports"),
("boxing","sports"),
("cycling","sports"),

# food
("pizza","food"),
("burger","food"),
("pasta","food"),
("bread","food"),
("cheese","food"),
("rice","food"),
("soup","food"),
("salad","food"),
("chocolate","food"),
("coffee","food"),

# materials
("wood","materials"),
("metal","materials"),
("plastic","materials"),
("glass","materials"),
("stone","materials"),
("steel","materials"),
("copper","materials"),
("silver","materials"),
("gold","materials"),
("iron","materials"),

# sizes
("big","sizes"),
("large","sizes"),
("huge","sizes"),
("small","sizes"),
("tiny","sizes"),
("giant","sizes"),
("massive","sizes"),
("mini","sizes"),
("narrow","sizes"),
("wide","sizes"),

# colors
("red","colors"),
("blue","colors"),
("green","colors"),
("yellow","colors"),
("purple","colors"),
("black","colors"),
("white","colors"),
("orange","colors"),
("pink","colors"),
("brown","colors"),

# analogy_words
("king","royalty"),
("queen","royalty"),
("prince","royalty"),
("princess","royalty"),
("man","gender"),
("woman","gender"),
]

# words = [word for word, group in items]
# embeddings = encode(words)
#
# with open("../../embedding-projector-standalone/oss_data/embeddings.tsv", "w", encoding="utf-8") as f_emb, \
#      open("../../embedding-projector-standalone/oss_data/metadata.tsv", "w", encoding="utf-8") as f_meta:
#
#     f_meta.write("word\tgroup\n")
#
#     for (word, group), emb in zip(items, embeddings):
#         f_emb.write("\t".join(str(x.item()) for x in emb) + "\n")
#         f_meta.write(f"{word}\t{group}\n")


from sklearn.decomposition import PCA
import torch
import torch.nn.functional as F

words = [word for word, group in items]
embeddings = encode(words)   # shape: [N, 384]

# PCA to 10 dims
pca = PCA(n_components=10)
reduced = pca.fit_transform(embeddings.numpy())

# normalize again after reduction
reduced = torch.tensor(reduced, dtype=torch.float32)
reduced = F.normalize(reduced, p=2, dim=1)

with open("../../embedding-projector-standalone/oss_data/embeddings.tsv", "w", encoding="utf-8") as f_emb, \
     open("../../embedding-projector-standalone/oss_data/metadata.tsv", "w", encoding="utf-8") as f_meta:

    f_meta.write("word\tgroup\n")

    for (word, group), emb in zip(items, reduced):
        f_emb.write("\t".join(str(x.item()) for x in emb) + "\n")
        f_meta.write(f"{word}\t{group}\n")